In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# ==========================================
# 1. LOAD & ADJUST DATA
# ==========================================
df_synthetic = pd.read_csv('../data/processed/girona_for_rent_synthetic.csv')
df_real = pd.read_csv('../data/processed/girona_for_rent.csv')

def adjust_price_to_2026(row):
    year = row.get('year_available', 2026)
    price = row['price']
    factors = {2015: 1.45, 2016: 1.40, 2017: 1.30, 2018: 1.25, 2019: 1.20,
               2020: 1.20, 2021: 1.15, 2022: 1.10, 2023: 1.05}
    return price * factors.get(year, 1.0)

df_synthetic['price'] = df_synthetic.apply(adjust_price_to_2026, axis=1)

# Keep lat/lon this time!
common_cols = ['price', 'area', 'rooms', 'floors', 'elevator', 'lat', 'lon', 'census_tract']
df_synthetic = df_synthetic[common_cols].copy()
df_real = df_real[common_cols].copy()

df_synthetic['is_synthetic'] = 1
df_real['is_synthetic'] = 0

df_all = pd.concat([df_synthetic, df_real], axis=0).reset_index(drop=True)

# Safety Clean
df_all['census_tract'] = df_all['census_tract'].astype('category')
df_all['elevator'] = df_all['elevator'].fillna(0).astype(int)
df_all['rooms'] = df_all['rooms'].replace(0, 1)

# Logic Fix: Ground floor elevator = 0
df_all.loc[df_all['floors'] == 0, 'elevator'] = 0

# ==========================================
# 2. FEATURE ENGINEERING (The "Blurred" Fix)
# ==========================================
# 1. Round Coordinates to 3 decimals (~110m precision)
# This prevents memorizing exact synthetic points but keeps "micro-neighborhood" info.
df_all['lat_grid'] = df_all['lat'].round(3)
df_all['lon_grid'] = df_all['lon'].round(3)

# 2. Standard Luxury Signals
df_all['m2_per_room'] = df_all['area'] / df_all['rooms']
df_all['vertical_score'] = df_all['floors'] * df_all['elevator']
df_all['is_large'] = (df_all['area'] > 120).astype(int)

# 3. Coordinate Interaction (Crucial for location value)
df_all['area_lat'] = df_all['area'] * df_all['lat_grid']
df_all['area_lon'] = df_all['area'] * df_all['lon_grid']

# ==========================================
# 3. SPLIT & WEIGHT
# ==========================================
df_syn_clean = df_all[df_all['is_synthetic'] == 1]
df_real_clean = df_all[df_all['is_synthetic'] == 0]

real_train, real_test = train_test_split(df_real_clean, test_size=0.2, random_state=42)
train_final = pd.concat([df_syn_clean, real_train], axis=0).sample(frac=1, random_state=42)
test_final = real_test

# Features: Note we use lat_grid/lon_grid now
features = [
    'area', 'rooms', 'floors', 'elevator', 
    'census_tract', 'lat_grid', 'lon_grid',
    'm2_per_room', 'vertical_score', 'is_large', 'area_lat', 'area_lon'
]

X_train = train_final[features]
y_train = train_final['price']
X_test = test_final[features]
y_test = test_final['price']

# Tiered Weights
def get_weight(row):
    if row['is_synthetic'] == 1: return 1
    if row['price'] > 2000: return 50
    return 10

weights_train = train_final.apply(get_weight, axis=1)

# ==========================================
# 4. TRAIN
# ==========================================
dtrain = xgb.DMatrix(X_train, label=np.log1p(y_train), weight=weights_train, enable_categorical=True)
dtest = xgb.DMatrix(X_test, label=np.log1p(y_test), enable_categorical=True)

params = {
    'max_depth': 8,
    'eta': 0.02,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
}

model = xgb.train(
    params,
    dtrain,
    num_boost_round=5000,
    evals=[(dtrain, 'train'), (dtest, 'eval')],
    early_stopping_rounds=150,
    verbose_eval=500
)

# ==========================================
# 5. PREDICT
# ==========================================
log_preds = model.predict(dtest)
raw_preds = np.expm1(log_preds) 

# Safety Net
final_preds = np.where(raw_preds > 1800, raw_preds * 1.10, raw_preds)

mae = mean_absolute_error(y_test, final_preds)
print(f"\nFinal MAE on REAL DATA: €{mae:.2f}")

comparison = pd.DataFrame({'Actual': y_test, 'Predicted': final_preds, 'Error': final_preds - y_test})
print(comparison.sort_values(by='Error').head(10))

model.save_model("../data/model_results/girona_rent_model_v3_hybrid.json")

[0]	train-rmse:0.42432	eval-rmse:0.41478
[500]	train-rmse:0.07561	eval-rmse:0.13570
[730]	train-rmse:0.06174	eval-rmse:0.13705

Final MAE on REAL DATA: €224.24
      Actual    Predicted       Error
2080  3300.0  3111.574951 -188.425049
2038  1100.0   974.740173 -125.259827
2014  1650.0  1583.546387  -66.453613
2058  1500.0  1498.449585   -1.550415
2082  2700.0  2738.464111   38.464111
2032  1500.0  1573.142456   73.142456
2002   750.0   851.921265  101.921265
2070   640.0   744.127563  104.127563
2098  1150.0  1263.778809  113.778809
2020  1300.0  1423.427979  123.427979


In [6]:
import json

# Calculate the "Standard Stats" for each Census Tract from your training data
# We get the average location and the most common building type (floors/elevator)
tract_stats = df_all.groupby('census_tract')[['lat_grid', 'lon_grid', 'floors', 'elevator']].mean()

# Convert to dictionary
# We round elevator to 0 or 1 (integers)
tract_lookup = tract_stats.to_dict(orient='index')
for key in tract_lookup:
    tract_lookup[key]['elevator'] = int(round(tract_lookup[key]['elevator']))
    tract_lookup[key]['floors'] = int(round(tract_lookup[key]['floors']))

# Save to file
with open('../data/model_results/girona_rent_model_v3_hybrid_stats.json', 'w') as f:
    json.dump(tract_lookup, f)

print("Smart Lookup Table saved.")

Smart Lookup Table saved.
